# CSI Based WiFi Setting


### By Seth Kantz and Matthew Delafield

The goal for this project will be to detect motion in a room given CSI data from a raspberry pi 4b



### Imports

In [15]:
from pathlib import Path
import numpy as np
import re
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models


###

### Set Variables / Paths

In [16]:
#

ROOT = Path("CSV_Files/room4_iqbal")

STATIC_MAG   = ROOT / "Static_noperson_channel_11_20mhz" / "csi_magnitude_data"
DYNAMIC_MAG  = ROOT / "Dynamic_1_person_ch11_20mhz"      / "csi_magnitude_data"

STATIC_CPLX  = ROOT / "Static_noperson_channel_11_20mhz" / "csi_complex_data"
DYNAMIC_CPLX = ROOT / "Dynamic_1_person_ch11_20mhz"      / "csi_complex_data"

SUBCARRIERS = 49
L = 1024  # choose based on typical rows per file

### Helper Functions
#

In [17]:

def gather_paths(mode="magnitude"):
    # just using magnitude for now, can move to complex later
    if mode == "magnitude":
        static_files  = sorted(STATIC_MAG.glob("*.csv"))
        dynamic_files = sorted(DYNAMIC_MAG.glob("*.csv"))
    elif mode == "complex":
        static_files  = sorted(STATIC_CPLX.glob("*.csv"))
        dynamic_files = sorted(DYNAMIC_CPLX.glob("*.csv"))
    else:
        raise ValueError("mode must be 'magnitude' or 'complex'")

    paths = static_files + dynamic_files
    labels = np.array([0]*len(static_files) + [1]*len(dynamic_files), dtype=np.int32)
    return paths, labels

def load_mag_csv(path, L=L):
    df = pd.read_csv(path)
    X = df.to_numpy(dtype=np.float32)          # (T, 49)
    X = np.log1p(np.abs(X))                    # stabilize (if already magnitude, abs is harmless)
    T, S = X.shape
    assert S == SUBCARRIERS, f"Expected {SUBCARRIERS} cols, got {S}"

    if T >= L:
        start = np.random.randint(0, T - L + 1)
        X = X[start:start+L]
    else:
        X = np.pad(X, ((0, L-T), (0,0)), mode="constant")

    # per-file normalization
    X = (X - X.mean()) / (X.std() + 1e-6)
    return X[..., None]  # (L, 49, 1)

def recording_id(p: Path):
    # grabs naming for csi_YYYYMMDD_HHMMSS
    m = re.search(r"(csi_\d{8}_\d{6})", p.name)
    return m.group(1) if m else p.stem

def batch_generator(paths, labels, batch_size=8):
    paths = np.array(paths)
    labels = np.array(labels)
    n = len(paths)
    while True:
        idx = np.random.permutation(n)
        for i in range(0, n, batch_size):
            b = idx[i:i+batch_size]
            Xb = np.stack([load_mag_csv(paths[j]) for j in b])
            yb = labels[b]
            yield Xb, yb

    

### Read in Data
#

In [18]:
paths, labels = gather_paths(mode="magnitude")
print(len(paths), labels.mean())

paths, labels = gather_paths("magnitude")
groups = np.array([recording_id(p) for p in paths])

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss.split(paths, labels, groups=groups))

# optional val split from train
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=43)
train_idx2, val_idx = next(gss2.split(np.array(paths)[train_idx], labels[train_idx], groups=groups[train_idx]))
train_idx = train_idx[train_idx2]

train_paths = np.array(paths)[train_idx]
val_paths   = np.array(paths)[val_idx]
test_paths  = np.array(paths)[test_idx]

train_y = labels[train_idx]
val_y   = labels[val_idx]
test_y  = labels[test_idx]


95 0.8947368421052632


### Set Train / Test / Validation Split
#

In [19]:
#
batch_size = 8
train_gen = batch_generator(train_paths, train_y, batch_size)
val_gen   = batch_generator(val_paths, val_y, batch_size)
test_gen   = batch_generator(val_paths, test_y, batch_size)




### Configure 2D Convolutional neural net
#

In [20]:
inp = layers.Input(shape=(L, SUBCARRIERS, 1))
x = layers.Conv2D(16, (7,3), padding="same", activation="relu")(inp)
x = layers.MaxPool2D((4,2))(x)
x = layers.Conv2D(32, (7,3), padding="same", activation="relu")(x)
x = layers.MaxPool2D((4,2))(x)
x = layers.Conv2D(64, (5,3), padding="same", activation="relu")(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inp, out)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)


model.summary()



### Train Model

In [21]:
#
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", patience=8, mode="max", restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", factor=0.5, patience=3, mode="max", min_lr=1e-6),
]

model.fit(
    train_gen,
    steps_per_epoch=max(1, len(train_paths)//batch_size),
    validation_data=val_gen,
    validation_steps=max(1, len(val_paths)//batch_size),
    epochs=60,
    callbacks=callbacks
)


Epoch 1/60
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 353ms/step - accuracy: 0.8281 - auc: 0.6295 - loss: 0.4040 - val_accuracy: 0.8750 - val_auc: 1.0000 - val_loss: 0.3262 - learning_rate: 0.0010
Epoch 2/60
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step - accuracy: 0.8833 - auc: 0.7385 - loss: 0.3267 - val_accuracy: 0.8750 - val_auc: 1.0000 - val_loss: 0.2287 - learning_rate: 0.0010
Epoch 3/60
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 244ms/step - accuracy: 0.8833 - auc: 1.0000 - loss: 0.2103 - val_accuracy: 0.8750 - val_auc: 1.0000 - val_loss: 0.1936 - learning_rate: 0.0010
Epoch 4/60
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 241ms/step - accuracy: 0.9000 - auc: 1.0000 - loss: 0.1732 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.0929 - learning_rate: 0.0010
Epoch 5/60
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 241ms/step - accuracy: 0.9833 - auc: 1.0000 - loss: 0.0979 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.0724 - learning_rate: 5.0000e-04
Epoch 6/60
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 248ms/step - accuracy: 0.9833 - auc: 1.0000 - l

### Evaluate Model

In [ ]:
#
print(model.evaluate(test_gen))


   6748/Unknown 1372s 203ms/step - accuracy: 1.0000 - auc: 0.0000e+00 - loss: 0.0468

### Calculate errors / performance

In [ ]:
#

#

### Graph Data

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()
#